# Quantum Potential Dataset Generator

> This notebook defines analytical and numerical potential models, builds sparse measurement features, and generates training/holdout datasets for neural networks that reconstruct potential and probability-density curves.

## What this notebook does
- Implements 14 one-dimensional quantum potentials (training and holdout families).
- Solves each Hamiltonian numerically with finite differences.
- Normalizes wavefunctions and returns $\rho=|\psi|^2$.
- Creates sparse observations from full potential curves.
- Builds sequence features for model input and target tensors for supervised learning.

## Main outputs
- `x_train`: input feature tensor `[batch, points, channels]`.
- `v_train`: ground-truth potential tensor.
- `rho_train`: ground-truth probability density tensor.
- `labels_train`: potential-family label for each sample.

In [1]:
# External imports
from __future__ import annotations
from neural_nets import *

import argparse
import copy
import json
import random
from contextlib import nullcontext
from dataclasses import asdict, dataclass
from pathlib import Path
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, random_split, DataLoader

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

try:
    from kan import KAN as PyKAN
except Exception:  # pragma: no cover - optional dependency
    PyKAN = None

## Import Layer and Runtime Dependencies

This code cell defines the runtime foundation for the notebook.

### Main groups of imports
- Core Python tools: argument parsing, JSON, dataclasses, path handling, randomness, and copying.
- Scientific stack: NumPy for numerical arrays and algebra.
- Deep learning stack: PyTorch tensors, neural network modules, and functional API.
- Data utilities: TensorDataset, random_split, and DataLoader for mini-batch pipelines.
- Plotting: Matplotlib for optional diagnostics and visualization.
- Optional model backend: `PyKAN` is imported defensively; if unavailable, it is set to `None`.

### Why this matters
- The potential solvers rely on NumPy linear algebra routines.
- The dataset pipeline exports tensors directly for PyTorch training.
- Optional dependencies are handled without breaking notebook execution.

In [2]:
#Potentials

_LAST_RANDOM_DRAWS = {}


def _sample_uniform(value, default_range):
    if value is None:
        lo, hi = default_range
        return float(np.random.uniform(lo, hi))
    if isinstance(value, (tuple, list, np.ndarray)) and len(value) == 2:
        return float(np.random.uniform(value[0], value[1]))
    return float(value)


def _sample_int(value, default_range):
    if value is None:
        lo, hi = default_range
        return int(np.random.randint(lo, hi + 1))
    if isinstance(value, (tuple, list, np.ndarray)) and len(value) == 2:
        lo, hi = int(value[0]), int(value[1])
        return int(np.random.randint(lo, hi + 1))
    return int(value)


def _sample_choice(value, default_values):
    if value is None:
        return np.random.choice(default_values)
    if isinstance(value, (tuple, list, np.ndarray)):
        return np.random.choice(value)
    return value


def _is_variable_source(value):
    return value is None or isinstance(value, (tuple, list, np.ndarray))


def _draw_non_repeating(function_name, raw_sources, draw_fn, max_tries=32):
    """Draw random params and avoid repeating the same random tuple consecutively per function."""
    has_randomness = any(_is_variable_source(v) for v in raw_sources)
    if not has_randomness:
        return draw_fn()

    previous = _LAST_RANDOM_DRAWS.get(function_name)
    current = draw_fn()
    tries = 0
    while previous is not None and current == previous and tries < max_tries:
        current = draw_fn()
        tries += 1

    _LAST_RANDOM_DRAWS[function_name] = current
    return current

def normalize_wavefunction(psi, x):
    """
    |psi(x)|² dx = 1.
    """
    norm = np.sqrt(np.trapezoid(psi**2, x))
    return psi / norm


def normalized(value):
    v_min = np.min(value)
    v_max = np.max(value)

    if v_max == v_min:
        return np.zeros_like(value)

    return (value - v_min) / (v_max - v_min)



# Physics potentials to model training

# 1. Particle in the Box
def particle_in_box(
    x_p_min: float = None,
    x_p_max: float = None,
    n_point: int = None,
    n_states: int = None
    ):
    """
    Solves the Schrödinger equation for a particle in a 1D box.
    
    Args:
        x_p_min: Left boundary of the box
        x_p_max: Right boundary of the box
        n_point: Number of discretization points
        n_states: Quantum state index (0, 1, 2, ...)
    
    Returns:
        x: Position array (interior points only)
        V: Potential energy (zero inside box)
        energies: Energy eigenvalue
        psi: Normalized wavefunction
        rho: Probability density |psi|²
    """
    
    raw_sources = (x_p_min, x_p_max, n_point, n_states)

    def _draw_params():
        return (
            0.0 if x_p_min is None else float(x_p_min),
            1.0 if x_p_max is None else float(x_p_max),
            200 if n_point is None else int(n_point),
            _sample_int(n_states, (0, 4)),
        )

    x_p_min, x_p_max, n_point, n_states = _draw_non_repeating(
        "particle_in_box", raw_sources, _draw_params
    )

    # Create full grid including boundaries
    x_p_full = np.linspace(x_p_min, x_p_max, n_point)

    # Remove boundaries: psi(x_min) = 0 and psi(x_max) = 0 (boundary conditions)
    x = x_p_full[1:-1]

    # Discretization step
    dx = x[1] - x[0]
    
    # Potential inside the box is zero (infinite walls at boundaries)
    V = np.zeros_like(x)
    
    # Number of interior points
    N = len(x)
    
    # Kinetic energy operator: -1/2 d²/dx² (finite difference discretization)
    main_diag = np.full(N, 1.0 / dx**2)
    off_diag = np.full(N - 1, -0.5 / dx**2)
    
    # Construct kinetic energy matrix (tridiagonal)
    T = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)
    
    # Hamiltonian is just the kinetic energy (V=0 inside box)
    H = T
    
    # Solve eigenvalue problem: H psi = E psi
    eigenvalues, eigenvectors = np.linalg.eigh(H)
    
    # Extract energy and wavefunction for the requested state
    n_states = min(max(0, n_states), len(eigenvalues) - 1)
    energies = eigenvalues[n_states]
    psi = eigenvectors[:, n_states]
    
    # Normalize wavefunction to satisfy ∫|psi|² dx = 1
    psi = normalize_wavefunction(psi, x)
    
    # Probability density
    rho = psi**2

    V = normalized(V)  # Normalize potential for plotting
        
    return x, V, energies, psi, rho


# 2. Harmonic Oscillator
def harmonic_oscillator_wavefunction(
    x_h_min: float = None,
    x_h_max: float = None,
    n_point: int = None,
    omega: float = None,
    n_states: int = None
    ):

    raw_sources = (x_h_min, x_h_max, n_point, omega, n_states)

    def _draw_params():
        return (
            -5.0 if x_h_min is None else float(x_h_min),
            5.0 if x_h_max is None else float(x_h_max),
            200 if n_point is None else int(n_point),
            _sample_uniform(omega, (1.0, 5.0)),
            _sample_int(n_states, (0, 4)),
        )

    x_h_min, x_h_max, n_point, omega, n_states = _draw_non_repeating(
        "harmonic_oscillator_wavefunction", raw_sources, _draw_params
    )

  
    x_full = np.linspace(x_h_min, x_h_max, n_point)

    x_h = x_full[1:-1]

    dx = x_h[1] - x_h[0]

    #potential 
    V_h = 0.5 * omega**2 * x_h**2

    #Cinético (resolvendo por diferenças finitas)

    N = len(x_h)

    dx = x_h[1] - x_h[0]

    main_diag = np.full(N, 1.0 / dx**2)

    off_diag = np.full(N - 1, -0.5 / dx**2)

    T = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

    V_h_matrix = np.diag(V_h)

    H_h = T + V_h_matrix

    eigenvalues_h, engenvectors_h = np.linalg.eigh(H_h)

    n_states = min(max(0, n_states), len(eigenvalues_h) - 1)
    energies = eigenvalues_h[n_states]

    psi_h = engenvectors_h[:, n_states]

    psi_h_normalized = normalize_wavefunction(psi_h, x_h)

    rho_h = psi_h_normalized**2

    V = normalized(V_h)

    return x_h, V, energies, psi_h_normalized, rho_h


# 3. Meio oscilador Hamônico
def half_harmonic_oscillator(
    x_hal_min: float = None,
    x_hal_max: float = None,
    omega: float = None,
    n_states: int = None,
    n_points: int = None
    ):

    raw_sources = (x_hal_min, x_hal_max, n_points, omega, n_states)

    def _draw_params():
        return (
            0.0 if x_hal_min is None else float(x_hal_min),
            5.0 if x_hal_max is None else float(x_hal_max),
            200 if n_points is None else int(n_points),
            _sample_uniform(omega, (1.0, 5.0)),
            _sample_int(n_states, (0, 4)),
        )

    x_hal_min, x_hal_max, n_points, omega, n_states = _draw_non_repeating(
        "half_harmonic_oscillator", raw_sources, _draw_params
    )

    x_full = np.linspace(x_hal_min, x_hal_max, n_points)

    x_hal = x_full[1:-1]

    dx = x_hal[1] - x_hal[0]

    V_hal = 0.5 * omega**2 * x_hal**2

    N = len(x_hal)

    main_diag = np.full(N, 1.0 / dx**2)

    off_diag = np.full(N - 1, -0.5 / dx**2)

    T = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

    V_hal_matrix = np.diag(V_hal)

    H_hal = T + V_hal_matrix

    eigenvalues_h, engenvectors_h = np.linalg.eigh(H_hal)

    n_states = min(max(0, n_states), len(eigenvalues_h) - 1)
    energies = eigenvalues_h[n_states]

    psi_h = engenvectors_h[:, n_states]

    psi_h_normalized = normalize_wavefunction(psi_h, x_hal)

    rho_h = psi_h_normalized**2

    V = normalized(V_hal)

    return x_hal, V, energies, psi_h_normalized, rho_h


# 4. Morse Potential

def morse(
    x_min: float = None,
    x_max: float = None,
    n_point: int = None,
    D: float = None,
    a: float = None,
    r_0: float = None,
    n_states: int = None
):
    raw_sources = (x_min, x_max, n_point, D, a, r_0, n_states)

    def _draw_params():
        return (
            0.1 if x_min is None else float(x_min),
            5.0 if x_max is None else float(x_max),
            200 if n_point is None else int(n_point),
            float(_sample_choice(D, (2.0, 5.0, 8.0))),
            float(_sample_choice(a, (1.0, 2.0))),
            1.0 if r_0 is None else float(r_0),
            _sample_int(n_states, (0, 4)),
        )

    x_min, x_max, n_point, D, a, r_0, n_states = _draw_non_repeating(
        "morse", raw_sources, _draw_params
    )

    r_m_full = np.linspace(x_min, x_max, n_point)

    r_m = r_m_full[1:-1]

    V_m = D * (1 - np.exp(-a * (r_m - r_0))) ** 2

    N = len(r_m)

    dx = r_m[1] - r_m[0]

    main_diag = np.full(N, 1.0 / dx**2)

    off_diag = np.full(N - 1, -0.5 / dx**2)

    T = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

    V_m_matrix = np.diag(V_m)

    H_m = T + V_m_matrix

    eigenvalues_h, engenvectors_h = np.linalg.eigh(H_m)

    n_states = min(max(0, n_states), len(eigenvalues_h) - 1)
    energies = eigenvalues_h[n_states]

    psi_m = engenvectors_h[:, n_states]

    psi_m_normalized = normalize_wavefunction(psi_m, r_m)

    rho_m = psi_m_normalized**2

    V = normalized(V_m)

    return r_m, V, energies, psi_m_normalized, rho_m


# 5. Poschl-Teller hiperbolico
def solve_poschl_teller_h(
    x_min: float = None,
    x_max: float = None,
    n_points: int = None,
    alpha: float = None,
    lam: float = None,
    n_states: int = None,
    ):
    """
    resolve H psi = E psi para:
        H = T + V
    com:
        T = -1/2 d²/dx² 
    """

    raw_sources = (x_min, x_max, n_points, alpha, lam, n_states)

    def _draw_params():
        x_min_v = -5.0 if x_min is None else float(x_min)
        x_max_v = 5.0 if x_max is None else float(x_max)
        n_points_v = 200 if n_points is None else int(n_points)
        alpha_v = _sample_uniform(alpha, (0.4, 2.0))
        lam_v = int(_sample_choice(lam, (1, 2, 3, 4)))
        if n_states is None:
            n_states_v = int(np.random.randint(0, max(1, lam_v)))
        else:
            n_states_v = _sample_int(n_states, (0, 4))
            n_states_v = min(max(0, n_states_v), max(0, lam_v - 1))
        return (x_min_v, x_max_v, n_points_v, alpha_v, lam_v, n_states_v)

    x_min, x_max, n_points, alpha, lam, n_states = _draw_non_repeating(
        "solve_poschl_teller_h", raw_sources, _draw_params
    )

    x_full = np.linspace(x_min, x_max, n_points)

    # Remover os pontos das bordas (boundary points)
    x = x_full[1:-1]
    #x = x_full

    #Menor passo entre dois pontos - Correspondente diferencial
    dx = x[1] - x[0]

    # Número de pontos de discretização para construção da matriz T
    N = len(x)

    # Calculo da secante hip
    sech_ax = 1 / np.cosh(alpha * x)

    # Calculo do potencial V(x)
    prefactor = -0.5 * alpha**2 * lam * (lam + 1)
    V = prefactor * sech_ax**2

    # Definir o operador cinetico = -1/2 d²/dx² - Diagonal principal
    main_diag = np.full(N, 1.0 / dx**2)

    # preencher as duas diagonais vizinhas da matriz do operador cinético
    off_diag = np.full(N - 1, -0.5 / dx**2)

    # Matriz de energia cinética (Hamiltoniano cinético)
    T = (np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1))

    # Para dar calcular as matrizes T + V, preciso Trans V m matriz
    V_matrix = np.diag(V)

    # Hamiltoniano total
    H = T + V_matrix

    #generalizando estados de energia do sistema:
    eigenvalues, eigenvectors = np.linalg.eigh(H)

    n_states = min(max(0, n_states), len(eigenvalues) - 1)
    energies = eigenvalues[n_states]
    psi_states = eigenvectors[:, n_states]

    psi_states = normalize_wavefunction(psi_states, x)

    rho = psi_states**2

    V = normalized(V)  # Normalize potential for plotting

    return x, V, energies, psi_states, rho


# 6. Poschl-Teller Trigonometrico

def solve_poschl_teller_t(
        n_states: int = None,
        n_points: int = None,
        alpha: float = None,
        A: float = None,
        B: float = None,
        ):

    raw_sources = (n_states, n_points, alpha, A, B)

    def _draw_params():
        return (
            _sample_int(n_states, (0, 4)),
            200 if n_points is None else int(n_points),
            _sample_uniform(alpha, (0.4, 2.0)),
            _sample_uniform(A, (1.0, 5.0)),
            _sample_uniform(B, (1.0, 5.0)),
        )

    n_states, n_points, alpha, A, B = _draw_non_repeating(
        "solve_poschl_teller_t", raw_sources, _draw_params
    )

    # liminte inferior tem que ser maior que zero 
    eps_bound = 1e-4

    x_t = np.linspace(eps_bound, (np.pi / (2 * alpha)) - eps_bound, n_points)
    dx = x_t[1] - x_t[0]
    x = x_t[1:-1]

    prefactor = A*(A-1) * (1 / np.sin(alpha * x)**2) + B*(B-1) * (1 / np.cos(alpha * x)**2)
    V = alpha**2 * prefactor / 2

    N = len(x)

    # Definir o operador cinetico = -1/2 d²/dx² - Diagonal principal
    main_diag = np.full(N, 1.0 / dx**2)
    # preencher as duas diagonais vizinhas da matriz do operador cinético
    off_diag = np.full(N - 1, -0.5 / dx**2)

    # Matriz de energia cinética (Hamiltoniano cinético)
    T = (np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1))
    # Para dar calcular as matrizes T + V, preciso Trans V m matriz
    V_matrix = np.diag(V)

    # Hamiltoniano total
    H = T + V_matrix

    #generalizando estados de energia do sistema:
    eigenvalues, eigenvectors = np.linalg.eigh(H)

    n_states = min(max(0, n_states), len(eigenvalues) - 1)
    energies = eigenvalues[n_states]
    psi_states = eigenvectors[:, n_states]

    psi_states = normalize_wavefunction(psi_states, x)
    rho = psi_states**2

    V = normalized(V) 

    return x, V, energies, psi_states, rho


# 7 . Radial coulomb potential
def coulomb_radial_effective_potential(
        r_min: float = None,
        r_max: float = None,
        Z: float = None,
        ell: int = None,
        n_states: int = None,
        n_points: int = None
    ):

    raw_sources = (r_min, r_max, n_points, n_states, Z, ell)

    def _draw_params():
        r_min_v = 1e-3 if r_min is None else float(r_min)
        r_max_v = 10.0 if r_max is None else float(r_max)
        n_points_v = 200 if n_points is None else int(n_points)
        n_states_v = _sample_int(n_states, (0, 4))
        Z_v = _sample_uniform(Z, (1.0, 5.0))
        if ell is None:
            ell_v = int(np.random.randint(0, n_states_v + 1))
        else:
            ell_v = min(_sample_int(ell, (0, 4)), n_states_v)
        return (r_min_v, r_max_v, n_points_v, n_states_v, Z_v, ell_v)

    r_min, r_max, n_points, n_states, Z, ell = _draw_non_repeating(
        "coulomb_radial_effective_potential", raw_sources, _draw_params
    )

    r_full = np.linspace(r_min, r_max, n_points)

    dr = r_full[1] - r_full[0]

    # Remove boundaries:
    # u(r_min) = 0 and u(r_max) = 0
    r = r_full[1:-1]

    V_eff = (-Z / r + ell * (ell + 1) / (2 * r**2))

    N = len(r)

    main_diagonal = np.full(N, 1.0 / dr**2)

    off_diagonal = np.full(N - 1, -0.5 / dr**2)

    T = (
        np.diag(main_diagonal)
        + np.diag(off_diagonal, k=1)
        + np.diag(off_diagonal, k=-1)
    )

    V_matrix = np.diag(V_eff)

    H = T + V_matrix

    eigenvalues, eigenvectors = np.linalg.eigh(H)

    n_states = min(max(0, n_states), len(eigenvalues) - 1)
    energies = eigenvalues[n_states]

    # First eigenvector = radial reduced wavefunction u(r)
    u_0 = eigenvectors[:, n_states]

    # Normalize:
    u_0 = normalize_wavefunction(u_0, r)

    # Radial probability density
    rho_radial_0 = u_0**2

    V_eff = normalized(V_eff)

    return r, V_eff, energies, u_0, rho_radial_0

# 8. Radial Oscillator Potential
def radial_oscillator_potential(
        r_min: float = None,
        r_max: float = None,
        omega: float = None,
        g: float = None,
        n_states: int = None,
        n_points: int = None
):

    raw_sources = (r_min, r_max, n_points, omega, g, n_states)

    def _draw_params():
        return (
            0.0 if r_min is None else float(r_min),
            10.0 if r_max is None else float(r_max),
            200 if n_points is None else int(n_points),
            _sample_uniform(omega, (1.0, 5.0)),
            _sample_uniform(g, (0.0, 1.0)),
            _sample_int(n_states, (0, 4)),
        )

    r_min, r_max, n_points, omega, g, n_states = _draw_non_repeating(
        "radial_oscillator_potential", raw_sources, _draw_params
    )

    r = np.linspace(r_min, r_max, n_points)

    dr = r[1] - r[0]
    r = r[1:-1]

    V = 0.5 * omega**2 * r**2 + g / r**2

    N = len(r)

    main_diagonal = np.full(N, 1.0 / dr**2)

    off_diagonal = np.full(N - 1, -0.5 / dr**2)

    T = (
        np.diag(main_diagonal)
        + np.diag(off_diagonal, k=1)
        + np.diag(off_diagonal, k=-1)
    )

    V_matrix = np.diag(V)

    H = T + V_matrix

    eigenvalues, eigenvectors = np.linalg.eigh(H)

    n_states = min(max(0, n_states), len(eigenvalues) - 1)
    energies = eigenvalues[n_states]
    psi = eigenvectors[:, n_states]

    psi = normalize_wavefunction(psi, r)

    rho = psi**2

    V = normalized(V)

    return r, V, energies, psi, rho


# 9. Kratzer potential
def solve_kratzer_potential(
    x_min: float = None,
    x_max: float = None,
    D_e: float = None,
    r_0: float = None,
    n_states: int = None,
    n_points: int = None
):
    raw_sources = (x_min, x_max, n_points, D_e, r_0, n_states)

    def _draw_params():
        return (
            0.1 if x_min is None else float(x_min),
            5.0 if x_max is None else float(x_max),
            200 if n_points is None else int(n_points),
            float(_sample_choice(D_e, (2.0, 5.0, 8.0))),
            1.0 if r_0 is None else float(r_0),
            _sample_int(n_states, (0, 4)),
        )

    x_min, x_max, n_points, D_e, r_0, n_states = _draw_non_repeating(
        "solve_kratzer_potential", raw_sources, _draw_params
    )

    x_full_k = np.linspace(x_min, x_max, n_points)

    x_kr = x_full_k[1:-1]

    V_kr = D_e * ((x_kr - r_0) / x_kr) ** 2

    N = len(x_kr)

    dx = x_kr[1] - x_kr[0]

    main_diagonal = np.full(N, 1.0 / dx**2)

    off_diagonal = np.full(N - 1, -0.5 / dx**2)

    T = (
        np.diag(main_diagonal)
        + np.diag(off_diagonal, k=1)
        + np.diag(off_diagonal, k=-1)
    )

    V_kr_matrix = np.diag(V_kr)

    H = T + V_kr_matrix

    eigenvalues, eigenvectors = np.linalg.eigh(H)

    n_states = min(max(0, n_states), len(eigenvalues) - 1)
    energies = eigenvalues[n_states]
    psi = eigenvectors[:, n_states]

    psi = normalize_wavefunction(psi, x_kr)

    rho = psi**2

    V = normalized(V_kr)

    return x_kr, V, energies, psi, rho

# Phisics potentials model to test

# 10. Hulthen Potential

def hulthen_potential(
    x_min: float = None,
    x_max: float = None,
    n_point: int = None,
    V_0 : float = None,
    n_states: int = None,
    a: float = None
    ):

    raw_sources = (x_min, x_max, n_point, V_0, a, n_states)

    def _draw_params():
        return (
            0.01 if x_min is None else float(x_min),
            5.0 if x_max is None else float(x_max),
            200 if n_point is None else int(n_point),
            _sample_uniform(V_0, (2.0, 8.0)),
            float(_sample_choice(a, (1.0, 2.0))),
            _sample_int(n_states, (0, 4)),
        )

    x_min, x_max, n_point, V_0, a, n_states = _draw_non_repeating(
        "hulthen_potential", raw_sources, _draw_params
    )

    x_full = np.linspace(x_min, x_max, n_point)

    x_hu = x_full[1:-1]

    dx = x_hu[1] - x_hu[0]
    
    exp_term = np.exp(-a * x_hu)
    V_hu = -V_0 * exp_term / (1 - exp_term)

    N = len(x_hu)

    main_diag = np.full(N, 1.0 / dx**2)

    off_diag = np.full(N - 1, -0.5 / dx**2)

    T = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

    V_h_matrix = np.diag(V_hu)

    H_h = T + V_h_matrix

    eigenvalues_h, engenvectors_h = np.linalg.eigh(H_h)

    n_states = min(max(0, n_states), len(eigenvalues_h) - 1)
    energies = eigenvalues_h[n_states]

    psi_h = engenvectors_h[:, n_states]

    psi_h_normalized = normalize_wavefunction(psi_h, x_hu)

    rho_h = psi_h_normalized**2

    V = normalized(V_hu)

    return x_hu, V, energies, psi_h_normalized, rho_h

# 11. Rose-Morse Potential II

def rose_morse(
    x_min: float = None,
    x_max: float = None,
    n_point: int = None,
    A: int =  None,
    B: int = None,
    n_states: int = None
    ):

    raw_sources = (x_min, x_max, n_point, A, B, n_states)

    def _draw_params():
        return (
            -5.0 if x_min is None else float(x_min),
            5.0 if x_max is None else float(x_max),
            200 if n_point is None else int(n_point),
            _sample_uniform(A, (1.0, 5.0)),
            _sample_uniform(B, (1.0, 5.0)),
            _sample_int(n_states, (0, 4)),
        )

    x_min, x_max, n_point, A, B, n_states = _draw_non_repeating(
        "rose_morse", raw_sources, _draw_params
    )

    x_rm_full = np.linspace(x_min, x_max, n_point)

    x_rm = x_rm_full[1:-1]

    dx = x_rm[1] - x_rm[0]

    secante = 1 / (np.cosh(x_rm))

    V_rm = - A * (A + 1) * secante ** 2 + 2 * B * np.tanh(x_rm)

    N = len(x_rm)

    main_diag = np.full(N, 1.0 / dx**2)

    off_diag = np.full(N - 1, -0.5 / dx**2)

    T = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

    V_h_matrix = np.diag(V_rm)

    H_h = T + V_h_matrix

    eigenvalues_h, engenvectors_h = np.linalg.eigh(H_h)

    n_states = min(max(0, n_states), len(eigenvalues_h) - 1)
    energies = eigenvalues_h[n_states]

    psi_h = engenvectors_h[:, n_states]

    psi_h_normalized = normalize_wavefunction(psi_h, x_rm)

    rho_h = psi_h_normalized**2

    V = normalized(V_rm)

    return x_rm, V, energies, psi_h_normalized, rho_h

# 12. Scarf II potential

def scar_II (
    x_min: float = None,
    x_max: float = None,
    A: int = None,
    B: int = None,
    n_point: int = None,
    n_states: int = None
    ):

    raw_sources = (x_min, x_max, n_point, A, B, n_states)

    def _draw_params():
        return (
            -5.0 if x_min is None else float(x_min),
            5.0 if x_max is None else float(x_max),
            200 if n_point is None else int(n_point),
            _sample_uniform(A, (1.0, 5.0)),
            _sample_uniform(B, (1.0, 5.0)),
            _sample_int(n_states, (0, 4)),
        )

    x_min, x_max, n_point, A, B, n_states = _draw_non_repeating(
        "scar_II", raw_sources, _draw_params
    )

    x_sc_full = np.linspace(x_min, x_max, n_point)

    x_sc = x_sc_full[1:-1]

    secante = 1 / (np.cosh(x_sc)) 

    V_sc = -(A * (A + 1) + B**2) * secante**2 + B * (2 * A + 1) * secante * np.tanh(x_sc)

    dx = x_sc[1] - x_sc[0] 

    N = len(x_sc)

    main_diag = np.full(N, 1.0 / dx**2)

    off_diag = np.full(N - 1, -0.5 / dx**2)

    T = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

    V_h_matrix = np.diag(V_sc)

    H_h = T + V_h_matrix

    eigenvalues_h, engenvectors_h = np.linalg.eigh(H_h)

    n_states = min(max(0, n_states), len(eigenvalues_h) - 1)
    energies = eigenvalues_h[n_states]

    psi_h = engenvectors_h[:, n_states]

    psi_h_normalized = normalize_wavefunction(psi_h, x_sc)

    rho_h = psi_h_normalized**2

    V = normalized(V_sc)

    return x_sc, V, energies, psi_h_normalized, rho_h


# 13. Eckart potential

def eckart (
    x_min: float = None,
    x_max: float = None,
    A: int = None,
    B: int = None,
    n_point: int = None,
    n_states: int = None
):
    raw_sources = (x_min, x_max, n_point, A, B, n_states)

    def _draw_params():
        return (
            0.01 if x_min is None else float(x_min),
            5.0 if x_max is None else float(x_max),
            200 if n_point is None else int(n_point),
            _sample_uniform(A, (1.0, 5.0)),
            _sample_uniform(B, (1.0, 5.0)),
            _sample_int(n_states, (0, 4)),
        )

    x_min, x_max, n_point, A, B, n_states = _draw_non_repeating(
        "eckart", raw_sources, _draw_params
    )

    x_ec_full = np.linspace(x_min, x_max, n_point)

    x_ec = x_ec_full[1:-1]

    dx = x_ec[1] - x_ec[0]

    cshc = 1 / np.sinh(x_ec)

    coth = np.cosh(x_ec) / np.sinh(x_ec)

    V_ec = A * (A-1) * cshc ** 2 - 2 * B * coth

    N = len(x_ec)
    main_diag = np.full(N, 1.0 / dx**2)

    off_diag = np.full(N - 1, -0.5 / dx**2)

    T = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

    V_h_matrix = np.diag(V_ec)

    H_h = T + V_h_matrix

    eigenvalues_h, engenvectors_h = np.linalg.eigh(H_h)

    n_states = min(max(0, n_states), len(eigenvalues_h) - 1)
    energies = eigenvalues_h[n_states]

    psi_h = engenvectors_h[:, n_states]

    psi_h_normalized = normalize_wavefunction(psi_h, x_ec)

    rho_h = psi_h_normalized**2

    V = normalized(V_ec)

    return x_ec, V, energies, psi_h_normalized, rho_h


# 14. Manning-Rosen
def manning_rose(
    x_min: float = None,
    x_max: float = None,
    A: int = None,
    B: int = None,
    a: int = None,
    n_point: int = None,
    n_states: int = None

):

    raw_sources = (x_min, x_max, n_point, A, B, a, n_states)

    def _draw_params():
        return (
            0.01 if x_min is None else float(x_min),
            5.0 if x_max is None else float(x_max),
            200 if n_point is None else int(n_point),
            _sample_uniform(A, (2.0, 10.0)),
            _sample_uniform(B, (1.0, 5.0)),
            float(_sample_choice(a, (1.0, 2.0))),
            _sample_int(n_states, (0, 4)),
        )

    x_min, x_max, n_point, A, B, a, n_states = _draw_non_repeating(
        "manning_rose", raw_sources, _draw_params
    )

    x_mr_full = np.linspace(x_min, x_max, n_point)

    x_mr = x_mr_full[1:-1]

    dx = x_mr[1] - x_mr[0]

    V_mr = (-A * np.exp(-a * x_mr)) / (1 - np.exp(-a * x_mr)) + (B * np.exp(-a * x_mr)) / (1 - np.exp(-a * x_mr)) ** 2 

    N = len(x_mr)

    main_diag = np.full(N, 1.0 / dx**2)

    off_diag = np.full(N - 1, -0.5 / dx**2)

    T = np.diag(main_diag) + np.diag(off_diag, k=1) + np.diag(off_diag, k=-1)

    V_h_matrix = np.diag(V_mr)

    H_h = T + V_h_matrix

    eigenvalues_h, engenvectors_h = np.linalg.eigh(H_h)

    n_states = min(max(0, n_states), len(eigenvalues_h) - 1)
    energies = eigenvalues_h[n_states]

    psi_h = engenvectors_h[:, n_states]

    psi_h_normalized = normalize_wavefunction(psi_h, x_mr)

    rho_h = psi_h_normalized**2

    V = normalized(V_mr)

    return x_mr, V, energies, psi_h_normalized, rho_h



if __name__ == "__main__":
    # Example usage of the potentials
    
    # 1. Particle in a Box
    x1, V1, energies1, psi1, rho1 = particle_in_box(n_states=0)

    # 2. Harmonic Oscillator
    x2, V2, energies2, psi2, rho2 = harmonic_oscillator_wavefunction(n_states=0)

    # 3. Half Harmonic Oscillator
    x3, V3, energies3, psi3, rho3 = half_harmonic_oscillator(n_states=0)

    # 4. Morse Potential
    x4, V4, energies4, psi4, rho4 = morse(n_states=0)

    # 5. Poschl-Teller Hyperbolic
    x5, V5, energies5, psi5, rho5 = solve_poschl_teller_h(n_states=0)

    # 6. Poschl-Teller Trigonometric
    x6, V6, energies6, psi6, rho6 = solve_poschl_teller_t(n_states=0)

    # 7. Radial Coulomb Potential
    x7, V7, energies7, psi7, rho7 = coulomb_radial_effective_potential(n_states=0)

    # 8. Radial Oscillator Potential
    x8, V8, energies8, psi8, rho8 = radial_oscillator_potential(n_states=0)

    # 9. Kratzer Potential
    x9, V9, energies9, psi9, rho9 = solve_kratzer_potential(n_states=0)

    # 10. Hulthen Potential
    x10, V10, energies10, psi10, rho10 = hulthen_potential(n_states=0)

    # 11. Rose-Morse Potential II
    x11, V11, energies11, psi11, rho11 = rose_morse(n_states=0)

    # 12. Scarf II Potential
    x12, V12, energies12, psi12, rho12 = scar_II(n_states=0)

    # 13. Eckart Potential
    x13, V13, energies13, psi13, rho13 = eckart(n_states=0)

    # 14. Manning-Rosen Potential
    x14, V14, energies14, psi14, rho14 = manning_rose(n_states=0)

## Full Reference: Potential and Solver Functions

This section documents every function defined in the potentials cell.

### Random-parameter helper functions
- `_sample_uniform(value, default_range)`: draws a float from a range or uses a fixed scalar.
- `_sample_int(value, default_range)`: draws an integer from a range or uses a fixed scalar.
- `_sample_choice(value, default_values)`: samples from a discrete set or returns a fixed value.
- `_is_variable_source(value)`: detects whether a parameter source is stochastic.
- `_draw_non_repeating(function_name, raw_sources, draw_fn, max_tries=32)`: avoids consecutive duplicate random tuples per function.

### Numerical normalization utilities
- `normalize_wavefunction(psi, x)`: scales wavefunction so that $\int |\psi(x)|^2 dx = 1$.
- `normalized(value)`: applies min-max normalization to `[0, 1]` with a zero-array fallback for constant signals.

### Physics potential generators
Each function returns `(x, V, energies, psi, rho)` and uses finite-difference diagonalization of the Hamiltonian.

- `particle_in_box(...)`: infinite-wall box with zero inner potential.
- `harmonic_oscillator_wavefunction(...)`: quadratic confining potential.
- `half_harmonic_oscillator(...)`: harmonic potential on a half-domain.
- `morse(...)`: anharmonic molecular-like Morse well.
- `solve_poschl_teller_h(...)`: hyperbolic Pöschl-Teller potential.
- `solve_poschl_teller_t(...)`: trigonometric Pöschl-Teller potential.
- `coulomb_radial_effective_potential(...)`: radial Coulomb plus centrifugal term.
- `radial_oscillator_potential(...)`: radial oscillator with inverse-square contribution.
- `solve_kratzer_potential(...)`: Kratzer molecular potential.
- `hulthen_potential(...)`: short-range Hulthén interaction.
- `rose_morse(...)`: Rosen-Morse type II form.
- `scar_II(...)`: Scarf II hyperbolic potential.
- `eckart(...)`: Eckart potential form.
- `manning_rose(...)`: Manning-Rosen potential.

### Demonstration block
The `if __name__ == "__main__":` block runs one sample call per potential family to validate outputs and provide quick sanity checks.

## Potential Models

> Each function below returns a tuple `(x, V, energy, psi, rho)` using a finite-difference Hamiltonian solver.

### Common behavior
- Inputs can be fixed scalars or random ranges/options.
- Random draws are de-duplicated to avoid identical consecutive parameter sets per function.
- `psi` is normalized with numerical integration.
- `V` is min-max normalized for stable downstream learning and plotting.

### Grouping used later in the notebook
- **Train families:** harmonic oscillator, Morse, Pöschl-Teller variants, Coulomb radial effective, radial oscillator, Kratzer, Rose-Morse, Scarf II.
- **Holdout families:** particle in a box, Hulthén, half harmonic oscillator, Eckart, Manning-Rosen.

In [3]:
# Potential-family registry used by dataset generation
"""
Each entry in TRAIN_FAMILIES and HOLDOUT_FAMILIES maps:
    (family_name, callable)
where callable(n_points, state) returns:
    x, V, energy, psi, rho

:param n_points: Number of spatial points in the discretization grid.
 :param state: Quantum state index used when selecting the eigenpair.

Example
-------
morse_func = next(func for name, func in TRAIN_FAMILIES if name == "morse")
curve = morse_func(n_points=100, state=0)

:returns curve tuple fields:
- curve[0]: spatial grid x
- curve[1]: potential values V(x)
- curve[2]: selected energy eigenvalue
- curve[3]: normalized wavefunction psi(x)
- curve[4]: probability density rho(x) = |psi(x)|^2
"""

TRAIN_FAMILIES = [
    ("harmonic_oscillator_wavefunction", lambda n_points, state: harmonic_oscillator_wavefunction(n_point=n_points, n_states=state)),
    ("morse", lambda n_points, state: morse(n_point=n_points, n_states=state)),
    ("solve_poschl_teller_h", lambda n_points, state: solve_poschl_teller_h(n_points=n_points, n_states=state)),
    ("solve_poschl_teller_t", lambda n_points, state: solve_poschl_teller_t(n_points=n_points, n_states=state)),
    ("coulomb_radial_effective_potential", lambda n_points, state: coulomb_radial_effective_potential(n_points=n_points, n_states=state)),
    ("radial_oscillator_potential", lambda n_points, state: radial_oscillator_potential(n_points=n_points, n_states=state)),
    ("solve_kratzer_potential", lambda n_points, state: solve_kratzer_potential(n_points=n_points, n_states=state)),
    ("rose_morse", lambda n_points, state: rose_morse(n_point=n_points, n_states=state)),
    ("scar_II", lambda n_points, state: scar_II(n_point=n_points, n_states=state)),
]

HOLDOUT_FAMILIES = [
    ("particle_in_box", lambda n_points, state: particle_in_box(n_point=n_points, n_states=state)),
    ("hulthen_potential", lambda n_points, state: hulthen_potential(n_point=n_points, n_states=state)),
    ("half_harmonic_oscillator", lambda n_points, state: half_harmonic_oscillator(n_points=n_points, n_states=state)),
    ("eckart", lambda n_points, state: eckart(n_point=n_points, n_states=state)),
    ("manning_rose", lambda n_points, state: manning_rose(n_point=n_points, n_states=state)),
]

In [4]:
def create_sparse_curve(curve: np.ndarray, measure_percent: float=15) -> tuple[np.ndarray, np.ndarray]:
    """Create a sparse observation of a dense curve and its measurement mask.

    The sampling strategy combines three criteria:
    1. Edge coverage (start/end regions).
    2. Uniform coverage over the full domain.
    3. High-gradient points (informative local changes).

    :param curve: Dense 1D signal to subsample.
    :param measure_percent: Percentage of points to keep as measured.
    :returns:
        sparse: Array with unmeasured points set to zero.
        mask: Binary array where 1.0 marks measured points.
    """

    lenght = len(curve)

    # Enforce at least two measured points.
    n_measured = max(2, int(round(lenght * measure_percent / 100)))
    # Never request more points than available.
    n_measured = min(lenght, n_measured)

    # Use fixed proportions for edge, uniform, and gradient-based picks.
    edge_count = min(lenght, max(2, n_measured // 6))  # Number of edge points to include.
    uniform_count = min(lenght, max(2, n_measured // 3))  # Number of uniformly distributed points.
    gradient_count = min(lenght, max(2, n_measured // 3)) # Number of gradient-priority points.

    # Edge points: first edge_count and last edge_count indices.
    edge_idx = np.r_[np.arange(edge_count), np.arange(lenght - edge_count, lenght)]

    # Middle points sampled uniformly.
    uniform_idx = np.linspace(0, lenght - 1, uniform_count, dtype=int)

    # Prioritize points with largest absolute gradient.
    gradient = np.abs(np.gradient(curve))
    gradient_idx = np.argsort(gradient)[-gradient_count:]  # Highest-gradient indices.


    # Candidate pool from edge, uniform, and gradient heuristics.
    candidate_idx = np.unique(np.r_[edge_idx, uniform_idx, gradient_idx])

    if len(candidate_idx) > n_measured:
        indices = np.random.choice(candidate_idx, size=n_measured, replace=False)
    else:
        remaining = np.setdiff1d(np.arange(lenght), candidate_idx)
        extra = np.random.choice(remaining, size=n_measured - len(candidate_idx), replace=False)
        indices = np.concatenate((candidate_idx, extra))

    # Build binary measurement mask.
    mask = np.zeros(lenght, dtype=np.float32)
    mask[indices] = 1.0

    sparse = curve.astype(np.float32) * mask

    return sparse, mask

In [5]:
# Interpolate sparce batch
def interpolate_sparse_batch(sparse: np.ndarray, V_mask: np.ndarray) -> np.ndarray:
    """
    Interpolates the sparse measurements in V_sparse using linear interpolation.
    The interpolation is performed only on the measured points indicated by V_mask.

    Args:
        V_sparse (np.ndarray): Sparse measurements of the potential.
        V_mask (np.ndarray): Mask indicating which points are measured (1) and which are not (0).

    Returns:
        np.ndarray: Interpolated values for all points in the potential.
    """
    lenght = len(sparse)
    x_grid = np.arange(lenght)
    measured_idx = np.where(V_mask == 1)[0]
    
    if len(measured_idx) == 0:
        return np.zeros(lenght, dtype=np.float32)
    elif len(measured_idx) == 1:
        return np.full(lenght, sparse[measured_idx[0]], dtype=np.float32)

    return np.interp(x_grid, measured_idx, sparse[measured_idx]).astype(np.float32)

In [6]:
def build_sequence_features(sparse: np.ndarray, mask: np.ndarray, use_interp_channel: bool = True) -> np.ndarray:
    # Features consumed by the context encoder:
    # [sparse_curve, mask, x_coordinate, interpolated_prior].
    x_coord = np.linspace(-1, 1, len(sparse), dtype=np.float32)

    if use_interp_channel:
        interp = interpolate_sparse_batch(sparse, mask)
        seq_feat = np.stack([sparse.astype(np.float32), mask.astype(np.float32), x_coord, interp], axis=-1)
        return seq_feat

    return np.stack([sparse.astype(np.float32), mask.astype(np.float32), x_coord], axis=-1)


def build_target_batch(target_v: torch.Tensor, use_interp_channel: bool = True) -> torch.Tensor:
    # Full target batch for the teacher encoder:
    # true curve at every point, all-ones mask, and prior equal to target.
    seq_len_local = target_v.shape[1]
    x_coord = torch.linspace(-1.0, 1.0, seq_len_local, device=target_v.device, dtype=target_v.dtype)
    x_coord = x_coord.unsqueeze(0).expand(target_v.shape[0], -1)
    ones = torch.ones_like(target_v)

    if use_interp_channel:
        return torch.stack((target_v, ones, x_coord, target_v), dim=-1)
    return torch.stack((target_v, ones, x_coord), dim=-1)


def split_features(features: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    # Returns sparse values, mask, and x coordinate channels.
    return features[:, :, 0], features[:, :, 1], features[:, :, 2]


def combine_with_input_prior(pred: torch.Tensor, features: torch.Tensor) -> torch.Tensor:
    # Important post-processing:
    # 1) blend prediction with interpolated prior
    # 2) enforce measured points to remain exactly equal to sparse input.
    if features.shape[-1] < 4:
        return pred

    sparse = features[:, :, 0]
    mask = features[:, :, 1]
    prior = features[:, :, 3]
    blended = 0.7 * pred + 0.3 * prior
    return torch.where(mask > 0.5, sparse, blended)


def derivative_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    # Penalizes mismatch in local slope between curves.
    return F.mse_loss(pred[:, 1:] - pred[:, :-1], target[:, 1:] - target[:, :-1])


def curvature_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    # Penalizes mismatch in discrete curvature (second derivative).
    pred_curv = pred[:, 2:] - 2.0 * pred[:, 1:-1] + pred[:, :-2]
    target_curv = target[:, 2:] - 2.0 * target[:, 1:-1] + target[:, :-2]
    return F.mse_loss(pred_curv, target_curv)


def weighted_reconstruction_loss(pred: torch.Tensor, target: torch.Tensor, features: torch.Tensor) -> torch.Tensor:
    # Reconstruction loss for V with:
    # - focus on high-variation regions
    # - separate weighting for measured/unmeasured points
    # - first- and second-derivative smoothness terms.
    _, mask, _ = split_features(features)

    grad = torch.zeros_like(target)
    local_diff = torch.abs(target[:, 1:] - target[:, :-1])
    grad[:, 1:] += local_diff
    grad[:, :-1] += local_diff

    weights = torch.clamp(1.0 + 4.0 * grad / (grad.mean(dim=1, keepdim=True) + 1e-8), 1.0, 12.0)
    error = (pred - target).pow(2)
    mse = torch.mean(weights * error)

    measured = torch.sum(mask * error) / (torch.sum(mask) + 1e-8)
    unmeasured = torch.sum((1.0 - mask) * error) / (torch.sum(1.0 - mask) + 1e-8)

    return 0.5 * mse + 1.5 * measured + 0.75 * unmeasured + 0.75 * derivative_loss(pred, target) + 0.2 * curvature_loss(pred, target)


def rho_reconstruction_loss(pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    # rho reconstruction loss with shape terms (derivative/curvature).
    return 1.5 * F.mse_loss(pred, target) + 0.5 * derivative_loss(pred, target) + 0.1 * curvature_loss(pred, target)

In [7]:
def generate_dataset(n_points_per_curve: int, samples_per_family: int, families: list[tuple[str, object]], seed: int=7, ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, list[str]]:
    """Generate supervised tensors from a list of potential families.

    For each family, this function repeatedly samples curves until it collects
    `samples_per_family` valid examples. A valid example must have expected
    length and finite numeric values for both potential and density.

    :param n_points_per_curve: Grid size requested from each family function.
    :param samples_per_family: Number of accepted samples to collect per family.
    :param families: List of `(family_name, callable)` where callable signature is
        `callable(n_points_per_curve, n_state)` and returns `(x, V, E, psi, rho)`.
    :param seed: Random seed for reproducible NumPy draws.
    :returns:
        x: Input feature tensor `[N, L, C]` (sparse + mask + coordinates + prior).
        v: Target potential tensor `[N, L]`.
        rho: Target probability-density tensor `[N, L]`.
        family_names: Label list with length `N`.
    :raises RuntimeError: If a family cannot produce enough valid samples.
    """
    np.random.seed(seed)

    expected_legnth = n_points_per_curve - 2

    n_range = list(range(1, 6))

    input_features: list[np.ndarray] = []
    target_v: list[np.ndarray] = []
    target_rho: list[np.ndarray] = []
    family_names: list[str] = []

    for family_name, family_func in families:
        collected = 0 
        attempts = 0

        max_attempts = samples_per_family * 10  # Safety cap to avoid infinite loops.

        while collected < samples_per_family and attempts < max_attempts:

            attempts += 1
            n_state = random.choice(n_range)

            try:
                _, potential, _, _, rho = family_func(n_points_per_curve, n_state)

            except Exception:
                continue

            if len(potential) != expected_legnth or len(rho) != expected_legnth:
                continue

            if not np.all(np.isfinite(potential)) or not np.all(np.isfinite(rho)):
                continue

            sparse, mask = create_sparse_curve(potential, measure_percent=15)
            input_features.append(build_sequence_features(sparse, mask, use_interp_channel=True))
            target_v.append(potential.astype(np.float32))
            target_rho.append(rho.astype(np.float32))
            family_names.append(family_name)

            collected += 1

        if collected != samples_per_family:
            raise RuntimeError(f"{family_name}: generated {collected}/{samples_per_family} valid samples")

    # Build tensors from collected samples.
    x = torch.tensor(np.stack(input_features), dtype=torch.float32)
    v = torch.tensor(np.stack(target_v), dtype=torch.float32)
    rho = torch.tensor(np.stack(target_rho), dtype=torch.float32)

    return x, v, rho, family_names

## Full Reference: Dataset Construction and Training Inputs

This section documents all remaining code blocks from family selection to dataloaders.

### Family registries
- `TRAIN_FAMILIES`: potential generators used to build in-distribution training data.
- `HOLDOUT_FAMILIES`: held-out potential generators for generalization tests.

### Sparse measurement and feature engineering
- `create_sparse_curve(curve, measure_percent)`: selects measured points and returns sparse signal + binary mask.
- `interpolate_sparse_batch(sparse, V_mask)`: linearly interpolates missing regions from measured samples.
- `build_sequence_features(sparse, mask, use_interp_channel=True)`: stacks model inputs as feature channels.
- `build_target_batch(target_v, use_interp_channel=True)`: creates teacher-style full-observation targets.
- `split_features(features)`: extracts sparse, mask, and coordinate channels.
- `combine_with_input_prior(pred, features)`: enforces measured points and blends prediction with interpolation prior.

### Loss definitions
- `derivative_loss(pred, target)`: aligns local slope behavior.
- `curvature_loss(pred, target)`: aligns second-derivative structure.
- `weighted_reconstruction_loss(pred, target, features)`: weighted potential reconstruction objective with smoothness terms.
- `rho_reconstruction_loss(pred, target)`: density reconstruction objective with derivative and curvature penalties.

### Dataset generation
- `generate_dataset(n_points_per_curve, samples_per_family, families, seed=7)`:
  validates generated curves, applies sparse sampling, builds feature tensors, and returns labels.

### Final execution cells
- Hyperparameter cell defines grid size, samples per family, and seed.
- Dataset generation cell builds `x_train`, `v_train`, `rho_train`, and `labels_train`.
- Final cell wraps tensors into `TensorDataset`, performs train/validation split, and creates `DataLoader` objects.

In [8]:
n_points_per_curve = 202

train_samples_per_family = 200
houldout_samples_per_family = 64

seed = 7

In [9]:
x_train, v_train, rho_train, labels_train = generate_dataset(n_points_per_curve, train_samples_per_family, TRAIN_FAMILIES, seed=seed)

In [10]:
dataset_train = TensorDataset(x_train, v_train, rho_train)
train_dataset = int(0.8 * len(dataset_train))
test_dataset = len(dataset_train) - train_dataset

train_ds, val_ds = random_split(
    dataset_train,
    [train_dataset, test_dataset],
    generator=torch.Generator().manual_seed(seed),
)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
test_loader = DataLoader(val_ds, batch_size=16, shuffle=False)